# Custom TensorFlow Tiny YOLO

This notebook implements a custom **Tiny YOLO** object detector using TensorFlow/Keras.

**Contents:**
1. Configuration
2. Model Architecture
3. Label Encoding
4. Custom YOLO Loss
5. Decode Predictions
6. Build & Compile

## 1. Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np

print(f"TensorFlow version: {tf.__version__}")

## 2. Configuration

Core hyperparameters for the model.

In [ ]:
IMAGE_SIZE = 416   
GRID_SIZE  = 13    
NUM_CLASSES = 1

print(f"Image size  : {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Grid size   : {GRID_SIZE}x{GRID_SIZE}")
print(f"Num classes : {NUM_CLASSES}")

## 3. Model Architecture

Tiny YOLO built from stacked `conv_block` units (Conv2D → BatchNorm → LeakyReLU) followed by MaxPooling layers.

In [ ]:
def conv_block(x, filters, kernel=3, stride=1):
    x = layers.Conv2D(filters, kernel, stride, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)
    return x


def build_tiny_yolo(input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)):
    inputs = layers.Input(shape=input_shape)

    x = conv_block(inputs, 16)
    x = layers.MaxPooling2D(2)(x)

    x = conv_block(x, 32)
    x = layers.MaxPooling2D(2)(x)

    x = conv_block(x, 64)
    x = layers.MaxPooling2D(2)(x)

    x = conv_block(x, 128)
    x = layers.MaxPooling2D(2)(x)

    x = conv_block(x, 256)
    x = layers.MaxPooling2D(2)(x)

    x = conv_block(x, 512)

    # Final 1x1 conv to predict (5 + NUM_CLASSES) values per cell
    output = layers.Conv2D(5 + NUM_CLASSES, 1, padding="same")(x)

    return Model(inputs, output)

## 4. Label Encoding

Converts raw bounding boxes into the YOLO grid target tensor.

In [ ]:
def encode_labels(boxes, classes):
    target = np.zeros((GRID_SIZE, GRID_SIZE, 5 + NUM_CLASSES))

    for box, cls in zip(boxes, classes):
        x, y, w, h = box

        grid_x = int(x * GRID_SIZE)
        grid_y = int(y * GRID_SIZE)

        if grid_x >= GRID_SIZE or grid_y >= GRID_SIZE:
            continue

        target[grid_y, grid_x, 0:4] = [x, y, w, h]  # box coords
        target[grid_y, grid_x, 4]   = 1.0            # objectness
        target[grid_y, grid_x, 5]   = cls            # class label

    return target


# --- Quick sanity check ---
sample_boxes   = [(0.5, 0.5, 0.2, 0.2)]
sample_classes = [0]
encoded = encode_labels(sample_boxes, sample_classes)
print(f"Target shape : {encoded.shape}")
print(f"Non-zero cells: {np.count_nonzero(encoded[..., 4])}")

## 5. Custom YOLO Loss

The loss has three components:
- **Box loss** – MSE on (x, y, w, h) for cells that contain an object.
- **Objectness loss** – MSE on the confidence score for all cells.
- **Class loss** – MSE on class probabilities for cells that contain an object.

In [ ]:
def yolo_loss(y_true, y_pred):
    obj_mask = y_true[..., 4:5]  

    box_loss = tf.reduce_sum(
        obj_mask * tf.square(y_true[..., 0:4] - y_pred[..., 0:4])
    )

    obj_loss = tf.reduce_sum(
        tf.square(y_true[..., 4:5] - y_pred[..., 4:5])
    )

    class_loss = tf.reduce_sum(
        obj_mask * tf.square(y_true[..., 5:] - y_pred[..., 5:])
    )

    total_loss = box_loss + obj_loss + class_loss
    return total_loss

## 6. Decode Predictions

Post-processing: filter grid cells by objectness threshold and return predicted boxes.

In [ ]:
def decode_predictions(pred, threshold=0.5):
    boxes = []

    for y in range(GRID_SIZE):
        for x in range(GRID_SIZE):
            cell = pred[y, x]
            if cell[4] > threshold:
                boxes.append(cell[0:4])

    return boxes

## 7. Build & Compile

Instantiate the model, attach the custom loss, and print a summary.

In [ ]:
model = build_tiny_yolo()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=yolo_loss
)

model.summary()

## 8. Quick Forward Pass Test

Verify that the model runs on a random input and produces the expected output shape.

In [ ]:
dummy_input = tf.random.normal((1, IMAGE_SIZE, IMAGE_SIZE, 3))
output = model(dummy_input, training=False)

expected_shape = (1, GRID_SIZE, GRID_SIZE, 5 + NUM_CLASSES)
print(f"Output shape   : {output.shape}")
print(f"Expected shape : {expected_shape}")
assert output.shape == expected_shape, "Shape mismatch!"
print("✓ Forward pass OK")